# [YOUR BENCHMARK NAME] — Task: `YOUR_TASK_NAME`

**Difficulty:** `medium` | **Task:** `YOUR_TASK_NAME`

Runs all models in one notebook save — output CSV contains every model's results.

---
## 1. Setup

In [ ]:
import json
from datetime import datetime
import kaggle_benchmarks as kbench
import pandas as pd

print(f'LLM       : {kbench.llm}')
print(f'Judge LLM : {kbench.judge_llm}')

---
## 2. Constants + Models

> **To see all available model slugs**, run the cell below before filling in `MODELS`.

In [ ]:
# Run this cell to discover all model slugs available on your benchmark token.
# Copy the slugs you want into the MODELS list in the next cell.
print('Available benchmark models:')
for slug in sorted(kbench.llms.keys()):
    print(f'  kbench.llms["{slug}"]')

In [ ]:
# ── Change this ──────────────────────────────────────────────────────
TASK_NAME = "YOUR_TASK_NAME"  # must match the %choose magic below

PROMPT_TEMPLATE = (
    'YOUR SYSTEM / TASK INSTRUCTIONS HERE.'
    '\n\n{question}\n\n'
    'YOUR OUTPUT INSTRUCTIONS HERE.'
)

TEMPERATURE_NOTE = 'Kaggle model proxy default'
# ──────────────────────────────────────────────────────────────────────

# ── Models to evaluate ────────────────────────────────────────────────
# Add or remove models here. Any model available on your Kaggle benchmark
# proxy token can be included. kbench runs every (model × question)
# combination automatically — one notebook save, one CSV, all models.
MODELS = [
    kbench.llms["google/gemini-2.5-flash"],
    kbench.llms["google/gemini-2.5-pro"],
    kbench.llms["meta/llama-4-maverick"],
    kbench.llms["anthropic/claude-sonnet-4"],
    kbench.llms["deepseek/deepseek-r1"],
]
# ──────────────────────────────────────────────────────────────────────

---
## 3. Evaluation Data

> `question` = raw question text only. The full prompt is built by `PROMPT_TEMPLATE.format(question=question)` inside the task.

In [ ]:
EVALUATION_DATA = [
    {
        'category':   'YOUR_CATEGORY',
        'difficulty': 'easy',          # easy | medium | hard
        'question':   'YOUR QUESTION TEXT HERE.',
        'answer':     'THE CORRECT ANSWER HERE.',
    },
    # Add more rows as needed …
]

df = pd.DataFrame(EVALUATION_DATA)
print(f'Loaded {len(df)} question(s)')
display(df.head())

---
## 4. Task Definition

- `_results` accumulates one dict per `(model × question)` combination
- Append fires **before** `assert_true` so failed runs are never dropped
- Criterion 0 → `answer_correct` | Criterion 1 → `reasoning_correct`

In [ ]:
_results = []  # accumulates (model × question) dicts

@kbench.task(name=TASK_NAME)
def evaluate(
    llm,
    question: str,
    answer: str,
    category: str = 'unknown',
    difficulty: str = 'medium',
):
    """YOUR_TASK_NAME"""
    prompt   = PROMPT_TEMPLATE.format(question=question)
    response = llm.prompt(prompt)

    # ── Define your evaluation criteria ───────────────────────────────
    # Criterion 0 is used as answer_correct, criterion 1 as reasoning_correct.
    # Add as many criteria as needed — all must pass for score=1.
    CRITERIA = [
        "CRITERION 0: describe what a correct answer must state.",
        "CRITERION 1: describe what correct reasoning must include.",
        "CRITERION 2 (optional): any additional constraint.",
    ]
    # ──────────────────────────────────────────────────────────────────

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=CRITERIA,
        response_text=response,
        judge_llm=kbench.judge_llm,
    )

    results_by_idx    = {i: r for i, r in enumerate(assessment.results)}
    answer_correct    = int(results_by_idx[0].passed) if 0 in results_by_idx else 0
    reasoning_correct = int(results_by_idx[1].passed) if 1 in results_by_idx else 0

    reasoning_notes = ' | '.join(
        f'[{r.criterion[:40]}]: {r.reason}'
        for r in assessment.results if not r.passed
    )

    system_part = PROMPT_TEMPLATE.split('{question}')[0].strip()
    messages = json.dumps([
        {'role': 'system',    'content': system_part},
        {'role': 'user',      'content': question},
        {'role': 'assistant', 'content': response},
    ], ensure_ascii=False)

    _results.append({
        'task_name':          TASK_NAME,
        'category':           category,
        'difficulty':         difficulty,
        'model_name':         str(llm),
        'model_version':      str(llm),
        'question':           question,
        'ground_truth':       answer,
        'prompt_template':    PROMPT_TEMPLATE,
        'messages':           messages,
        'llm_response':       response,
        'answer_correct':     answer_correct,
        'reasoning_correct':  reasoning_correct,
        'score':              int(all(r.passed for r in assessment.results)),
        'reasoning':          reasoning_notes,
        'judge_model':        str(kbench.judge_llm),
        'temperature_note':   TEMPERATURE_NOTE,
    })

    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f'[{category}/{difficulty}] {result.criterion}: {result.reason}',
        )

print(f'Task defined: {TASK_NAME}')

---
## 5. Run Evaluation

kbench runs every `(model × question)` combination automatically.
`_results` will have `len(MODELS) × len(df)` rows when done.

In [ ]:
results = evaluate.evaluate(llm=MODELS, evaluation_data=df)
print(results.as_dataframe().to_string())

---
## 6. Save Output CSV

One CSV — all models, all questions. This is what Evalflow pulls from the notebook output tab.

In [ ]:
results_df = pd.DataFrame(_results)
results_df['timestamp'] = datetime.now().isoformat()

model_slug  = "multi-model"
ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f'{TASK_NAME}__{model_slug}__{ts}.csv'

results_df.to_csv(output_path, index=False)
print(f'Saved  : {output_path}')
print(f'Rows   : {len(results_df)}')
print(f'Models : {results_df["model_version"].nunique()}')
print(f'Columns: {list(results_df.columns)}')
display(results_df.groupby("model_version")["score"].mean().rename("accuracy").reset_index())

---
## 7. Submit to Leaderboard

In [ ]:
%choose YOUR_TASK_NAME